In [ ]:
import pandas as pd
import json

## Execução original

In [ ]:
dataset_rows = []

with open('PublicHearingBR_LDS.jsonl', 'r') as f:
    for line in f:
        dataset_rows.append(json.loads(line))

dataset = pd.DataFrame(dataset_rows)

In [ ]:
dataset['len_transcricao'] = dataset['transcricao'].apply(lambda x: len(x))

In [ ]:
dataset['len_transcricao'].describe()

,len_transcricao
count,206.000000
mean,110937.509709
std,72353.900854
min,27145.000000
25%,74694.500000
50%,99444.500000
75%,128715.500000
max,881586.000000


### Extração de oradores e falas

In [ ]:
import re

padrao_orador = r"((?:O SR\.|A SRA\.)\s+(?:[\w\s\.\,\/\-]|(?:\([^\)]+\)))+?(?:\s[-–—]\s*|\s*[-–—]\s*))"

def extracao_blocos_fala(string):
  pedacos = re.split(padrao_orador, string)

  blocos_de_fala = []

  for i in range(1, len(pedacos), 2):
      orador = pedacos[i].strip().upper()
      orador = orador[:-1].strip()
      orador = ' '.join(orador.split()[2:])
      fala = [pedaco.strip() for pedaco in pedacos[i + 1].strip().split('\n') if len(pedaco.strip()) > 0]

      if fala:
          blocos_de_fala.append({
              "orador": orador,
              "fala": fala
          })

  return blocos_de_fala

#### Verificando se há duplicidade de oradores

In [ ]:
!pip install thefuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.6 MB/s eta 0:00:00


In [ ]:
from thefuzz import fuzz

def encontrar_quase_duplicados(termos, limite_similaridade = 70):
    duplicados = []
    termos_unicos = list(set([t.strip() for t in termos]))

    for i in range(len(termos_unicos)):
        for j in range(i + 1, len(termos_unicos)):
            termo1 = termos_unicos[i]
            termo2 = termos_unicos[j]

            similaridade = fuzz.ratio(termo1, termo2)

            if similaridade >= limite_similaridade:
                duplicados.append((termo1, termo2, f"{similaridade}%"))

    return duplicados

In [ ]:
todos_candidatos = set()
for transcricao in dataset['transcricao']:
  blocos_fala = extracao_blocos_fala(transcricao)
  oradores = set()
  for bloco in blocos_fala:
    oradores.add(bloco['orador'])
  candidatos = encontrar_quase_duplicados(oradores)
  todos_candidatos.update(candidatos)

In [ ]:
todos_candidatos

{('ADRIANA VENTURA(NOVO - SP)',
  'PRESIDENTE(ADRIANA VENTURA. NOVO - SP)',
  '78%'),
 ('AIRTON FALEIRO(BLOCO/PT - PA)', 'ALFREDINHO(BLOCO/PT - SP)', '70%'),
 ('ALBUQUERQUE(BLOCO/REPUBLICANOS - RR)',
  'ALUISIO MENDES(BLOCO/REPUBLICANOS - MA)',
  '75%'),
 ('ALBUQUERQUE(BLOCO/REPUBLICANOS - RR)',
  'PRESIDENTE(ALBUQUERQUE. BLOCO/REPUBLICANOS - RR)',
  '83%'),
 ('ALESSANDRA ALMEIDA', 'ALESSANDRA GOTTI', '71%'),
 ('ALEXANDRE DA SILVA', 'ALEXANDRE KALACHE', '74%'),
 ('AMARO NETO(BLOCO/REPUBLICANOS - ES)',
  'CARLOS GOMES(BLOCO/REPUBLICANOS - RS)',
  '81%'),
 ('AMÁLIA BARROS(PL - MT)', 'PRESIDENTE(AMÁLIA BARROS. PL - MT)', '75%'),
 ('ANA ANGÉLICA CAMPELO DE ALBUQUERQUE E MELO',
  'ANA ANGÉLICA CAMPELO DE ALBUQUERQUE',
  '91%'),
 ('ANA PAULA LIMA(BLOCO/PT - SC)',
  'PRESIDENTE(ANA PAULA LIMA. BLOCO/PT - SC)',
  '80%'),
 ('ANA PIMENTEL(BLOCO/PT - MG)', 'ANA PAULA LIMA(BLOCO/PT - SC)', '71%'),
 ('ANDRÉ FERNANDES(PL - CE)', 'PRESIDENTE(ANDRÉ FERNANDES. PL - CE)', '77%'),
 ('ARLINDO CHINAGLIA(BL

In [ ]:
mapeamentos = {
  ('ABILIO BRUNINI(PL - MT)', 'PRESIDENTE(ABILIO BRUNINI. PL - MT)', '76%'),
  ('ADRIANA VENTURA(NOVO - SP)',
  'PRESIDENTE(ADRIANA VENTURA. NOVO - SP)',
  '78%'),
  ('AMÁLIA BARROS(PL - MT)', 'PRESIDENTE(AMÁLIA BARROS. PL - MT)', '75%'),
  (
  'ANA ANGÉLICA CAMPELO DE ALBUQUERQUE',
  'ANA ANGÉLICA CAMPELO DE ALBUQUERQUE E MELO',
  '91%'),
  (
  'ANTONIO CARLOS RODRIGUES(PL - SP)',
  'PRESIDENTE(ANTONIO CARLOS RODRIGUES. PL - SP)',
  '82%'),
  (
  'ARNALDO JARDIM(BLOCO/CIDADANIA - SP)',
  'PRESIDENTE(ARNALDO JARDIM. BLOCO/CIDADANIA - SP)',
  '83%'),
  (
  'AUREO RIBEIRO(BLOCO/SOLIDARIEDADE - RJ)',
  'PRESIDENTE(AUREO RIBEIRO. BLOCO/SOLIDARIEDADE - RJ)',
  '84%'),
  (
  'CELSO RUSSOMANNO(BLOCO/REPUBLICANOS - SP)',
  'PRESIDENTE(CELSO RUSSOMANNO. BLOCO/REPUBLICANOS - SP)',
  '85%'),
  (
  'CORONEL CHRISÓSTOMO(PL - RO)',
  'PRESIDENTE(CORONEL CHRISÓSTOMO. PL - RO)',
  '79%'),
  (
  'DANYEL IORIO DE LIMA',
  'DANYEL IÓRIO DE LIMA',
  '95%'),
  (
  'DEBORA SIQUEIRA RAMOS BELTRAMM',
  'DEBORA SIQUEIRA RAMOS BELTRAMMI',
  '98%'),
  (
  'DENILDO RODRIGUES DE MORAIS (BIKO RODRIGUES)',
  'DENILDO RODRIGUES DE MORAES (BIKO RODRIGUES)',
  '98%'),
  (
  'ENIO VERRI(PT - PR)',
  'PRESIDENTE(ENIO VERRI. PT - PR)',
  '72%'),
  ('ERIKA KOKAY(BLOCO/PT - DF)',
  'PRESIDENTE(ERIKA KOKAY. BLOCO/PT - DF)',
  '78%'),
  ('EVAIR VIEIRA DE MELO(BLOCO/PP - ES)',
  'PRESIDENTE(EVAIR VIEIRA DE MELO. BLOCO/PP - ES)',
  '83%'),
  ('FILIPE MARTINS(PL - TO)', 'PRESIDENTE(FILIPE MARTINS. PL - TO)', '76%'),
  ('FÁBIO FÉLIX', 'FÁBIO FELIX', '91%'),
  ('GUTEMBERG REIS(BLOCO/MDB - RJ)',
  'PRESIDENTE(GUTEMBERG REIS. BLOCO/MDB - RJ)',
  '81%'),
  ('JUNINHO DO PNEU(BLOCO/UNIÃO - RJ)',
  'PRESIDENTE(JUNINHO DO PNEU. BLOCO/UNIÃO - RJ)',
  '82%'),
  ('LUIZIANNE LINS(BLOCO/PT - CE)',
  'PRESIDENTE(LUIZIANNE LINS. BLOCO/PT - CE)',
  '80%'),
  ('LUTIANA VALADARES F. BARBOSA',
  'LUTIANA VALADARES FERNANDES BARBOSA',
  '86%'),
  ('MARIA ELIZABETH DE OLIVEIRA (MÃE BETH DE OXUM)',
  'MARIA ELIZABETH SANTIAGO DE OLIVEIRA (MÃE BETH DE OXUM)',
  '91%'),
  ('MÁRCIO JERRY(BLOCO/PCDOB - MA)',
  'PRESIDENTE(MÁRCIO JERRY. BLOCO/PCDOB - MA)',
  '81%'),
  ('MÁRCIO TAVARES', 'MÁRCIO TAVARES SANTOS', '80%'),
  ('PADRE JOÃO(BLOCO/PT - MG)', 'PRESIDENTE(PADRE JOÃO. BLOCO/PT - MG)', '77%'),
  ('ABILIO BRUNINI(PL - MT)', 'PRESIDENTE(ABILIO BRUNINI. PL - MT)', '76%'),
  (
  'ALBUQUERQUE(BLOCO/REPUBLICANOS - RR)',
  'PRESIDENTE(ALBUQUERQUE. BLOCO/REPUBLICANOS - RR)',
  '83%'),
  (
  'ALCEU MOREIRA(BLOCO/MDB - RS)',
  'PRESIDENTE(ALCEU MOREIRA. BLOCO/MDB - RS)',
  '80%'),
  (
  'ANA PAULA LIMA(BLOCO/PT - SC)',
  'PRESIDENTE(ANA PAULA LIMA. BLOCO/PT - SC)',
  '80%'),
  (
  'ANDREIA SIQUEIRA(BLOCO/MDB - PA)',
  'PRESIDENTE(ANDREIA SIQUEIRA. BLOCO/MDB - PA)',
  '82%'),
  (
  'ANDRÉ FERNANDES(PL - CE)',
  'PRESIDENTE(ANDRÉ FERNANDES. PL - CE)',
  '77%'),
  (
  'ANTONIO BRITO(BLOCO/PSD - BA)',
  'PRESIDENTE(ANTONIO BRITO. BLOCO/PSD - BA)',
  '80%'),
  (
  'BIA KICIS(PL - DF)',
  'PRESIDENTE(BIA KICIS. PL - DF)',
  '71%'),
  (
  'BIBO NUNES(PL - RS)',
  'PRESIDENTE(BIBO NUNES. PL - RS)',
  '72%'),
  (
  'CARLOS GOMES(BLOCO/REPUBLICANOS - RS)',
  'PRESIDENTE(CARLOS GOMES. BLOCO/REPUBLICANOS - RS)',
  '84%'),
  (
  'CORONEL ULYSSES(BLOCO/UNIÃO - AC)',
  'PRESIDENTE(CORONEL ULYSSES. BLOCO/UNIÃO - AC)',
  '82%'),
  (
  'DANDARA(BLOCO/PT - MG)',
  'PRESIDENTE(DANDARA. BLOCO/PT - MG)',
  '75%'),
  (
  'EVAIR VIEIRA DE MELO(BLOCO/PP - ES)',
  'PRESIDENTE(EVAIR VIEIRA DE MELO. BLOCO/PP - ES)',
  '83%'),
  (
  'GERALDO RESENDE(BLOCO/PSDB - MS)',
  'PRESIDENTE(GERALDO RESENDE. BLOCO/PSDB - MS)',
  '82%'),
  (
  'GILVAN DA FEDERAL(PL - ES)',
  'PRESIDENTE(GILVAN DA FEDERAL. PL - ES)',
  '78%'),
  (
  'JORGE GOETTEN(PL - SC)',
  'PRESIDENTE(JORGE GOETTEN. PL - SC)',
  '75%'),
  (
  'LAFAYETTE DE ANDRADA(BLOCO/REPUBLICANOS - MG)',
  'PRESIDENTE(LAFAYETTE DE ANDRADA. BLOCO/REPUBLICANOS - MG)',
  '86%'),
  (
  'LEÔNIDAS CRISTINO(BLOCO/PDT - CE)',
  'PRESIDENTE(LEÔNIDAS CRISTINO. BLOCO/PDT - CE)',
  '82%'),
  (
  'LUIZ LIMA(PL - RJ)',
  'PRESIDENTE(LUIZ LIMA. PL - RJ)',
  '71%'),
  (
  'MARCEL VAN HATTEM(NOVO - RS)',
  'PRESIDENTE(MARCEL VAN HATTEM. NOVO - RS)',
  '79%'),
  (
  'MARX BELTRÃO(BLOCO/PP - AL)',
  'PRESIDENTE(MARX BELTRÃO. BLOCO/PP - AL)',
  '79%'),
  (
  'MURILLO GOUVEA(BLOCO/UNIÃO - RJ)',
  'PRESIDENTE(MURILLO GOUVEA. BLOCO/UNIÃO - RJ)',
  '82%'),
  (
  'NETO CARLETTO(BLOCO/PP - BA)',
  'PRESIDENTE(NETO CARLETTO. BLOCO/PP - BA)',
  '79%'),
  (
  'OSMAR TERRA(BLOCO/MDB - RS)',
  'PRESIDENTE(OSMAR TERRA. BLOCO/MDB - RS)',
  '79%'),
  (
  'PAULO GUEDES(BLOCO/PT - MG)',
  'PRESIDENTE(PAULO GUEDES. BLOCO/PT - MG)',
  '79%'),
  (
  'PROF. PAULO FERNANDO(BLOCO/REPUBLICANOS - DF)',
  'PRESIDENTE(PROF. PAULO FERNANDO. BLOCO/REPUBLICANOS - DF)',
  '86%'),
  (
  'RAIMUNDO SANTOS(BLOCO/PSD - PA)',
  'PRESIDENTE(RAIMUNDO SANTOS. BLOCO/PSD - PA)',
  '81%'),
  (
  'RICARDO SILVA(BLOCO/PSD - SP)',
  'PRESIDENTE(RICARDO SILVA. BLOCO/PSD - SP)',
  '80%'),
  (
  'SANDERSON(PL - RS)',
  'PRESIDENTE(SANDERSON. PL - RS)',
  '71%'),
  (
  'SILVIA CRISTINA(PL - RO)',
  'PRESIDENTE(SILVIA CRISTINA. PL - RO)',
  '77%'),
  (
  'PRESIDENTE(PROFESSORA LUCIENE CAVALCANTE. BLOCO/PSOL - SP)',
  'PROFESSORA LUCIENE CAVALCANTE(BLOCO/PSOL - SP)',
  '87%'),
  ('ROBINSON FARIA(PL - RN)', 'PRESIDENTE(ROBINSON FARIA. PL - RN)', '76%'),
  ('ROSÂNGELA REIS(PL - MG)', 'PRESIDENTE(ROSÂNGELA REIS. PL - MG)', '76%'),
  ('SEBASTIÂO COELHO', 'SEBASTIÃO COELHO',  '94%'),
  ('TALÍRIA PETRONE(PSOL - RJ)',
  'PRESIDENTE(TALÍRIA PETRONE. PSOL - RJ)',
  '78%'),
  ('TÂNIA SILVA SANTOS MADUREIRA', 'TANIA SILVA SANTOS MADUREIRA', '96%'),
  ('VITOR LIPPI(BLOCO/PSDB - SP)',
  'PRESIDENTE(VITOR LIPPI. BLOCO/PSDB - SP)',
  '79%'),
}

In [ ]:
mapeamentos_dict = {k[0]: k[1] for k in mapeamentos}

In [ ]:
blocos_fala_todos = []
for transcricao in dataset['transcricao']:
  blocos_fala = extracao_blocos_fala(transcricao)
  blocos_fala_final = []
  for bloco in blocos_fala:
    if bloco['orador'] in mapeamentos_dict:
      blocos_fala_final.append({ 'orador': mapeamentos_dict[bloco['orador']], 'fala': bloco['fala'] })
    else:
      blocos_fala_final.append(bloco)
  blocos_fala_todos.append(blocos_fala_final)

In [ ]:
dataset['blocos_fala'] = blocos_fala_todos

In [ ]:
dataset['oradores'] = dataset['blocos_fala'].apply(lambda x: sorted(list(set([y['orador'] for y in x]))))
dataset['n_oradores'] = dataset['oradores'].apply(lambda x: len(x))

In [ ]:
falas_completas = []

for blocos in blocos_fala_todos:
  todas_falas = []
  for bloco in blocos:
    for fala in bloco['fala']:
      todas_falas.append((bloco['orador'], fala))
  falas_completas.append(todas_falas)

In [ ]:
dataset['falas_completas'] = falas_completas

In [ ]:
dataset['falas_str'] = dataset['falas_completas'].apply(lambda falas: '\n'.join(['[{}] {}'.format(i + 1, falas[i][1]) for i in range(len(falas))]))

In [ ]:
from google.colab import userdata
api_key = userdata.get('EXPLANATION_OPENAI')

In [ ]:
PROMPT = """Você é um sistema especializado em Processamento de Linguagem Natural, focado em auditar e limpar transcrições de audiências públicas.
Sua tarefa é analisar o bloco de texto fornecido, onde cada bloco de fala é precedida por um identificador numérico e separado por quebra de linha.

Por exemplo:
[1] Bloco de fala 1.
[2] Bloco de fala 2.

Classifique o ID de cada bloco em uma das quatro categorias abaixo:

- CATEGORIA 1: PROTOCOLO - Saudações ("Bom dia a todos"), agradecimentos formais, composição de mesa, testes de áudio.
- CATEGORIA 2: GESTÃO DA AUDIÊNCIA: Passar ou pedir a palavra, controle de tempo, avisos administrativos, interrupções.
- CATEGORIA 3: DIGRESSÃO: Conversas paralelas, anedotas pessoais ou comentários fora do escopo do debate.
- CATEGORIA 4: CONTEÚDO RELEVANTE - Exposição de dados, apresentação de fatos, argumentações, defesas, propostas, dúvidas e respostas pertinentes ao tema central da audiência.

REGRA DE SEGURANÇA: Se um bloco contiver uma mistura de CONTEÚDO RELEVANTE e outras categorias ou se houver dúvida sobre sua relevância, classifique-a SEMPRE como CONTEÚDO RELEVANTE para evitar perda de contexto. Considere que, idealmente, os blocos sem CONTEÚDO RELEVANTE serão descartados a fim de gerar uma versão simplificada da audiência.

Retorne APENAS um objeto JSON válido, seguindo estritamente esta estrutura (não use intervalos como [1-3], liste cada número individualmente):

{{
  "ids_protocolo": [lista de inteiros com os IDs],
  "ids_gestao_da_audiencia": [lista de inteiros com os IDs],
  "ids_digressao": [lista de inteiros com os IDs],
  "ids_conteudo_relevante": [lista de inteiros com os IDs]
}}

[TEXTO DE ENTRADA]
{texto}"""

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key = api_key,
)

respostas = []
atual = 0

for transcricao in dataset['falas_str']:
  print('Transcrição #{}/{}...'.format(atual + 1, len(dataset)))
  atual += 1

  response = client.chat.completions.create(
      model = "gpt-5.6-luna",
      messages = [
          {"role": "user", "content": PROMPT.format(texto = transcricao)}
      ],
  )

  respostas.append(response)

Transcrição #1/206...
Transcrição #2/206...
Transcrição #3/206...
Transcrição #4/206...
Transcrição #5/206...
Transcrição #6/206...
Transcrição #7/206...
Transcrição #8/206...
Transcrição #9/206...
Transcrição #10/206...
Transcrição #11/206...
Transcrição #12/206...
Transcrição #13/206...
Transcrição #14/206...
Transcrição #15/206...
Transcrição #16/206...
Transcrição #17/206...
Transcrição #18/206...
Transcrição #19/206...
Transcrição #20/206...
Transcrição #21/206...
Transcrição #22/206...
Transcrição #23/206...
Transcrição #24/206...
Transcrição #25/206...
Transcrição #26/206...
Transcrição #27/206...
Transcrição #28/206...
Transcrição #29/206...
Transcrição #30/206...
Transcrição #31/206...
Transcrição #32/206...
Transcrição #33/206...
Transcrição #34/206...
Transcrição #35/206...
Transcrição #36/206...
Transcrição #37/206...
Transcrição #38/206...
Transcrição #39/206...
Transcrição #40/206...
Transcrição #41/206...
Transcrição #42/206...
Transcrição #43/206...
Transcrição #44/206.

In [ ]:
dataset['respostas'] = respostas

In [ ]:
dataset.to_csv('audiencias-simplificadas.csv')

## Análise dos resultados

In [ ]:
import csv
csv.field_size_limit(10000000)
simplificadas = pd.read_csv('audiencias-simplificadas.csv', engine = 'python')

In [ ]:
simplificadas.columns

Index(['id', 'materia', 'metadados', 'transcricao', 'len_transcricao',
       'blocos_fala', 'oradores', 'n_oradores', 'falas_completas',
       'len_transcricao_palavras', 'falas_str', 'respostas'],
      dtype='object')

In [ ]:
import re
import codecs

In [ ]:
def extrair_json_content(texto):
  if not isinstance(texto, str):
    return None

  match = re.search(
      r"content=['\"](\{.*?\})['\"],\s*(?:refusal=|role=|function_call=|tool_calls=)",
      texto,
      re.DOTALL,
  )

  if not match:
    match = re.search(
        r"content=['\"](.*?)['\"],\s*(?:refusal=|role=)", texto, re.DOTALL
    )

  if not match:
    return None

  conteudo_bruto = match.group(1)

  try:
    conteudo_limpo = (
        codecs.decode(conteudo_bruto, "unicode_escape")
        .encode("latin1")
        .decode("utf-8")
    )
  except Exception:
    conteudo_limpo = codecs.decode(conteudo_bruto, "unicode_escape")

  try:
    return json.loads(conteudo_limpo)
  except json.JSONDecodeError:
    return conteudo_limpo

In [ ]:
simplificadas['ids_classificados'] = simplificadas['respostas'].apply(lambda x: extrair_json_content(x))

In [ ]:
simplificadas['ids_classificados']

,ids_classificados
0,"{'ids_protocolo': [1, 2, 3, 5, 6, 22, 43, 78, ..."
1,"{'ids_protocolo': [1, 2, 32, 34, 41, 42, 46, 4..."
2,"{'ids_protocolo': [1, 2, 3, 4, 10, 11, 54, 60,..."
3,"{'ids_protocolo': [1, 5, 6, 13, 14, 15, 22, 23..."
4,"{'ids_protocolo': [1, 2, 3, 4, 5, 6, 10, 17, 1..."
...,...
201,"{'ids_protocolo': [3, 4, 21, 25, 27, 57, 58, 6..."
202,"{'ids_protocolo': [1, 3, 9, 10, 12, 28, 29, 30..."
203,"{'ids_protocolo': [1, 7, 8, 11, 14, 16, 17, 37..."
204,"{'ids_protocolo': [1, 16, 19, 22, 25, 27, 28, ..."


In [ ]:
simplificadas['ids_relevantes'] = simplificadas['ids_classificados'].apply(lambda x: x['ids_conteudo_relevante'])
simplificadas['ids_protocolo'] = simplificadas['ids_classificados'].apply(lambda x: x['ids_protocolo'])
simplificadas['ids_gestao_da_audiencia'] = simplificadas['ids_classificados'].apply(lambda x: x['ids_gestao_da_audiencia'])
simplificadas['ids_digressao'] = simplificadas['ids_classificados'].apply(lambda x: x['ids_digressao'])

In [ ]:
simplificadas['len_ids_relevantes'] = simplificadas['ids_relevantes'].apply(lambda x: len(x))
simplificadas['len_ids_protocolo'] = simplificadas['ids_protocolo'].apply(lambda x: len(x))
simplificadas['len_ids_gestao_da_audiencia'] = simplificadas['ids_gestao_da_audiencia'].apply(lambda x: len(x))
simplificadas['len_ids_digressao'] = simplificadas['ids_digressao'].apply(lambda x: len(x))

In [ ]:
simplificadas['n_turnos'] = simplificadas['falas_str'].apply(lambda x: len(x.split('\n')))

In [ ]:
simplificadas['taxa_relevante_turnos'] = simplificadas['len_ids_relevantes'] / simplificadas['n_turnos']
simplificadas['taxa_digressao_turnos'] = simplificadas['len_ids_digressao'] / simplificadas['n_turnos']
simplificadas['taxa_gestao_turnos'] = simplificadas['len_ids_gestao_da_audiencia'] / simplificadas['n_turnos']
simplificadas['taxa_protocolo_turnos'] = simplificadas['len_ids_protocolo'] / simplificadas['n_turnos']

In [ ]:
simplificadas['taxa_relevante_turnos'].describe()

,taxa_relevante_turnos
count,206.000000
mean,0.658236
std,0.074021
min,0.464061
25%,0.612517
50%,0.654882
75%,0.698904
max,1.000000


In [ ]:
simplificadas['taxa_digressao_turnos'].describe()

,taxa_digressao_turnos
count,206.000000
mean,0.012418
std,0.015329
min,0.000000
25%,0.002859
50%,0.007909
75%,0.015625
max,0.081081


In [ ]:
simplificadas['taxa_gestao_turnos'].describe()

,taxa_gestao_turnos
count,206.000000
mean,0.158902
std,0.058650
min,0.000000
25%,0.124693
50%,0.148956
75%,0.184175
max,0.459016


In [ ]:
simplificadas['taxa_protocolo_turnos'].describe()

,taxa_protocolo_turnos
count,206.000000
mean,0.166654
std,0.054732
min,0.000000
25%,0.131635
50%,0.171962
75%,0.207531
max,0.287671


taxa de simplificação:

In [ ]:
(1 - simplificadas['taxa_relevante_turnos']).describe()

,taxa_relevante_turnos
count,206.000000
mean,0.341764
std,0.074021
min,0.000000
25%,0.301096
50%,0.345118
75%,0.387483
max,0.535939


In [ ]:
import ast

In [ ]:
x = simplificadas['falas_completas'].iloc[0]
x_as_list = ast.literal_eval(x)

In [ ]:
simplificadas['falas_completas'] = simplificadas['falas_completas'].apply(lambda x: ast.literal_eval(x))

In [ ]:
simplificadas['ids_relevantes'] = simplificadas['ids_relevantes'].apply(lambda x: sorted(list(set(x))))

In [ ]:
simplificadas['falas_resumidas'] = simplificadas.apply(lambda row: [row['falas_completas'][i - 1] for i in row['ids_relevantes']], axis=1)

In [ ]:
def join_falas(falas):
  orador_anterior = ''
  string = ''

  for orador, fala in falas:
    if orador == orador_anterior:
      string += '\n\n' + fala
    else:
      string += '\n\n- {}:\n{}'.format(orador, fala)
      orador_anterior = orador

  return string

In [ ]:
simplificadas['falas_resumidas_str'] = simplificadas['falas_resumidas'].apply(lambda x: join_falas(x).strip())

In [ ]:
simplificadas[['id', 'falas_resumidas_str']].to_csv('simplificacao.csv', index = False)

In [ ]:
len(simplificadas['ids_protocolo'].iloc[0]) + len(simplificadas['ids_gestao_da_audiencia'].iloc[0]) + len(simplificadas['ids_digressao'].iloc[0]) + len(simplificadas['ids_relevantes'].iloc[0])

431

In [ ]:
simplificadas['falas_resumidas'] = simplificadas.apply(lambda row: '\n'.join([row['falas_str'].split('\n')[i - 1] for i in row['ids_relevantes']]), axis=1)

In [ ]:
simplificadas['len_total'] = simplificadas['falas_str'].apply(lambda x: len(x.split('\n')) + sum(len(y) for y in x.split('\n')))

In [ ]:
simplificadas['len_relevantes'] = simplificadas.apply(lambda row: sum(len(row['falas_str'].split('\n')[i - 1]) for i in row['ids_relevantes']), axis=1)

In [ ]:
simplificadas['len_digressao'] = simplificadas.apply(lambda row: sum(len(row['falas_str'].split('\n')[i - 1]) for i in row['ids_digressao']), axis=1)
simplificadas['len_gestao_da_audiencia'] = simplificadas.apply(lambda row: sum(len(row['falas_str'].split('\n')[i - 1]) for i in row['ids_gestao_da_audiencia']), axis=1)
simplificadas['len_protocolo'] = simplificadas.apply(lambda row: sum(len(row['falas_str'].split('\n')[i - 1]) for i in row['ids_protocolo']), axis=1)

In [ ]:
simplificadas['taxa_relevante'] = simplificadas['len_relevantes'] / simplificadas['len_total']
simplificadas['taxa_digressao'] = simplificadas['len_digressao'] / simplificadas['len_total']
simplificadas['taxa_gestao_da_audiencia'] = simplificadas['len_gestao_da_audiencia'] / simplificadas['len_total']
simplificadas['taxa_protocolo'] = simplificadas['len_protocolo'] / simplificadas['len_total']

In [ ]:
simplificadas['taxa_relevante'].describe()

,taxa_relevante
count,206.000000
mean,0.871304
std,0.036802
min,0.781025
25%,0.847241
50%,0.872960
75%,0.896460
max,0.996616


In [ ]:
simplificadas['taxa_digressao'].describe()

,taxa_digressao
count,206.000000
mean,0.007119
std,0.010062
min,0.000000
25%,0.000642
50%,0.004008
75%,0.009331
max,0.088883


In [ ]:
simplificadas['taxa_gestao_da_audiencia'].describe()

,taxa_gestao_da_audiencia
count,206.000000
mean,0.062078
std,0.024985
min,0.000000
25%,0.045311
50%,0.059515
75%,0.072021
max,0.180456


In [ ]:
simplificadas['taxa_protocolo'].describe()

,taxa_protocolo
count,206.000000
mean,0.054744
std,0.024264
min,0.000000
25%,0.038479
50%,0.050287
75%,0.069631
max,0.114905


Em média, 5% é protocolo, 6% é gestão da audiência e 0,7% é digressão e 87% é relevante.

Em turnos, digressão: 1%, gestão de turnos: 15%, protocolo: 17%, relevante: 66%.

In [ ]:
sum(simplificadas['taxa_digressao_turnos'] * simplificadas['n_turnos'])

889.0

In [ ]:
sum(simplificadas['taxa_gestao_turnos'] * simplificadas['n_turnos'])

11758.0

In [ ]:
sum(simplificadas['taxa_protocolo_turnos'] * simplificadas['n_turnos'])

10729.0

In [ ]:
sum(simplificadas['taxa_relevante_turnos'] * simplificadas['n_turnos'])

51711.0

In [ ]:
simplificadas.columns

Index(['id', 'materia', 'metadados', 'transcricao', 'len_transcricao',
       'blocos_fala', 'oradores', 'n_oradores', 'falas_completas',
       'len_transcricao_palavras', 'falas_str', 'respostas',
       'ids_classificados', 'ids_relevantes', 'ids_protocolo',
       'ids_gestao_da_audiencia', 'ids_digressao', 'len_ids_relevantes',
       'len_ids_protocolo', 'len_ids_gestao_da_audiencia', 'len_ids_digressao',
       'n_turnos', 'taxa_relevante_turnos', 'taxa_digressao_turnos',
       'taxa_gestao_turnos', 'taxa_protocolo_turnos', 'falas_resumidas',
       'falas_resumidas_str', 'len_total', 'len_relevantes', 'len_digressao',
       'len_gestao_da_audiencia', 'len_protocolo', 'taxa_relevante',
       'taxa_digressao', 'taxa_gestao_da_audiencia', 'taxa_protocolo'],
      dtype='object')

- 50 por classe (200 total)
- 2 anotadores por cada parágrafo
- se formos nós 4: cada um anota 100...

In [ ]:
# import ast
# ast.literal_eval(simplificadas['metadados'].iloc[0])

In [ ]:
temas_ids = {
    "Saúde pública e SUS": [6, 15, 28, 50, 56, 60, 62, 68, 76, 91, 105, 114, 123, 129, 140, 146, 157, 159, 164, 171, 181, 182, 185, 190, 199, 202, 204],
    "Educação, universidades e formação profissional": [18, 24, 31, 51, 92, 93, 111, 125, 137, 141, 151, 156, 160, 163, 168, 174, 184, 196],
    "Direitos das pessoas idosas": [9, 52, 63, 73, 110, 127, 149, 155],
    "Pessoas com deficiência, autismo e inclusão": [5, 12, 37, 53, 94, 96, 176, 200],
    "Mulheres e igualdade de gênero": [58, 89, 119, 121, 154, 165, 188, 203],
    "Crianças, adolescentes e juventude": [29, 30, 69, 74, 75, 84, 90, 100, 147],
    "Povos indígenas, quilombolas e tradicionais": [20, 82, 136, 139, 172],
    "Racismo, diversidade e direitos humanos": [16, 41, 48, 85, 170, 180, 187, 206],
    "Trabalho, previdência e FGTS": [3, 10, 19, 43, 55, 65, 66, 67, 72, 87, 112, 120, 128, 135, 142, 162],
    "Economia, tributação, crédito e seguros": [7, 8, 21, 26, 38, 79, 86, 117, 143, 144, 152, 153, 175, 193],
    "Energia e transição energética": [2, 23, 70, 71, 78, 104, 133, 134, 173, 178],
    "Meio ambiente e mudanças climáticas": [11, 14, 44, 115, 118, 158, 169, 183, 186],
    "Desastres ambientais e calamidades": [13, 49, 59, 107, 109, 113],
    "Transporte, rodovias e mobilidade": [36, 42, 77, 102, 131, 161, 177, 194],
    "Tecnologia, IA e proteção de dados": [27, 54, 88, 138, 166, 192, 197],
    "Internet, redes sociais e desinformação": [22, 33, 34, 46, 179, 189, 195],
    "Segurança pública, sistema prisional e crimes digitais": [57, 98, 103, 145, 201, 205],
    "Democracia, eleições e Judiciário": [1, 17, 35, 39, 80],
    "Cultura, esporte e turismo": [40, 47, 64, 101, 108, 130, 150],
    "Moradia, reforma agrária e regularização fundiária": [45, 83, 124],
    "Agricultura, mineração e indústria": [25, 61, 95, 97, 106, 122, 126],
    "Telecomunicações e radiodifusão": [4, 132],
    "Relações internacionais, guerras e refugiados": [32, 81, 99, 116, 198],
    "Proteção e bem-estar animal": [148, 167, 191],
}

id_para_tema = {id_aud: tema for tema, ids in temas_ids.items() for id_aud in ids}

simplificadas["tema_audiencia"] = simplificadas["id"].map(id_para_tema)

In [ ]:
minorias = [
    5, 9, 12, 20, 26, 29, 30, 32, 34, 37, 41, 48, 52, 53, 58, 60, 63, 73, 74, 83, 84, 85, 89, 90, 92, 94, 96, 99, 100, 110, 111, 119, 121, 127, 129, 136, 139, 147, 149, 154, 155, 160, 163, 165, 170, 172, 176, 180, 187, 188, 198, 203, 206
]

In [ ]:
simplificadas['is_minoria'] = simplificadas['id'].isin(minorias)

In [ ]:
data_classificacao = []

for index, row in simplificadas.iterrows():
    id_audiencia = row['id']
    ids_classificados = row['ids_classificados']

    for categoria, ids in ids_classificados.items():
        for id_frase in ids:
            data_classificacao.append({
                'id_audiencia': id_audiencia,
                'id_frase': id_frase,
                'categoria_frase': categoria,
                'tema_audiencia': row['tema_audiencia'],
                'is_minoria': id_audiencia in minorias
            })

classificacao_df = pd.DataFrame(data_classificacao)

In [ ]:
classificacao_df['categoria_frase'].value_counts()

,count
categoria_frase,
ids_conteudo_relevante,51711
ids_gestao_da_audiencia,11758
ids_protocolo,10729
ids_digressao,889


In [ ]:
amostra_df = (
    classificacao_df
    .groupby(['categoria_frase', 'is_minoria'], group_keys=False)
    .apply(lambda g: g.sample(n = 15, random_state = 13))
    .reset_index(drop=True)
)

/tmp/ipykernel_2987/2425154439.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n = 15, random_state = 13))


In [ ]:
import ast

def get_frase_text(row):
    id_aud = row['id_audiencia']
    id_fra = row['id_frase']

    falas_completa_str = simplificadas.loc[simplificadas['id'] == id_aud, 'falas_completas'].iloc[0]

    falas_completa_list = ast.literal_eval(falas_completa_str)

    if id_fra > 0 and id_fra <= len(falas_completa_list):
        return '- FRASE ANTERIOR:\n' + falas_completa_list[id_fra - 2][1] + '\n\n- FRASE PARA CLASSIFICAR:\n' + falas_completa_list[id_fra - 1][1] + '\n\n- FRASE POSTERIOR:\n' + (falas_completa_list[id_fra][1] if id_fra < len(falas_completa_list) else '')
    return None

amostra_df['frase'] = amostra_df.apply(get_frase_text, axis=1)
amostra_df.head()

In [ ]:
amostra_df.to_csv('amostra.csv', index = False)

In [ ]:
simplificadas['len_transcricao_palavras'] = simplificadas['transcricao'].apply(lambda x: len(x.split()))

In [ ]:
simplificadas['blocos_fala'] = simplificadas['blocos_fala'].apply(lambda x: ast.literal_eval(x))

In [ ]:
simplificadas['n_blocos_fala'] = simplificadas['blocos_fala'].apply(lambda x: len(x))

In [ ]:
simplificadas['len_transcricao_palavras'].describe()

In [ ]:
sum(simplificadas['n_blocos_fala'])

In [ ]:
simplificadas['n_oradores'].describe()

In [ ]:
simplificadas['n_blocos_fala'].describe()

In [ ]:
simplificadas['falas_completas'].apply(len).describe()

In [ ]:
(simplificadas['n_blocos_fala'] / simplificadas['n_oradores']).describe()

In [ ]:
sum(simplificadas['n_blocos_fala'])

In [ ]:
simplificadas.columns

Index(['id', 'materia', 'metadados', 'transcricao', 'len_transcricao',
       'blocos_fala', 'oradores', 'n_oradores', 'falas_completas',
       'len_transcricao_palavras', 'falas_str', 'respostas',
       'ids_classificados', 'ids_relevantes', 'ids_protocolo',
       'ids_gestao_da_audiencia', 'ids_digressao', 'len_ids_relevantes',
       'len_ids_protocolo', 'len_ids_gestao_da_audiencia', 'len_ids_digressao',
       'n_turnos', 'taxa_relevante_turnos', 'taxa_digressao_turnos',
       'taxa_gestao_turnos', 'taxa_protocolo_turnos', 'falas_resumidas',
       'falas_resumidas_str', 'len_total', 'len_relevantes', 'len_digressao',
       'len_gestao_da_audiencia', 'len_protocolo', 'taxa_relevante',
       'taxa_digressao', 'taxa_gestao_da_audiencia', 'taxa_protocolo',
       'tema_audiencia'],
      dtype='object')

In [ ]:
simplificadas[simplificadas['is_minoria']]['taxa_relevante'].describe()

,taxa_relevante
count,53.000000
mean,0.856794
std,0.033197
min,0.781025
25%,0.837154
50%,0.859097
75%,0.877151
max,0.939712


In [ ]:
simplificadas[simplificadas['is_minoria'] == False]['taxa_relevante'].describe()

,taxa_relevante
count,153.000000
mean,0.876331
std,0.036753
min,0.781295
25%,0.854534
50%,0.878111
75%,0.902538
max,0.996616


In [ ]:
simplificadas.columns

Index(['id', 'materia', 'metadados', 'transcricao', 'len_transcricao',
       'blocos_fala', 'oradores', 'n_oradores', 'falas_completas',
       'len_transcricao_palavras', 'falas_str', 'respostas',
       'ids_classificados', 'ids_relevantes', 'ids_protocolo',
       'ids_gestao_da_audiencia', 'ids_digressao', 'len_ids_relevantes',
       'len_ids_protocolo', 'len_ids_gestao_da_audiencia', 'len_ids_digressao',
       'n_turnos', 'taxa_relevante_turnos', 'taxa_digressao_turnos',
       'taxa_gestao_turnos', 'taxa_protocolo_turnos', 'falas_resumidas',
       'falas_resumidas_str', 'len_total', 'len_relevantes', 'len_digressao',
       'len_gestao_da_audiencia', 'len_protocolo', 'taxa_relevante',
       'taxa_digressao', 'taxa_gestao_da_audiencia', 'taxa_protocolo',
       'tema_audiencia', 'is_minoria'],
      dtype='object')

In [ ]:
from scipy import stats

# Separate the 'taxa_relevante' values for minority and non-minority groups
minority_relevance = simplificadas[simplificadas['is_minoria']]['taxa_relevante']
non_minority_relevance = simplificadas[simplificadas['is_minoria'] == False]['taxa_relevante']

# Perform independent samples t-test
t_statistic, p_value = stats.ttest_ind(minority_relevance, non_minority_relevance)

print(f"T-statistic: {t_statistic:.3f}")
print(f"P-value: {p_value:.3f}")

if p_value < 0.05:
    print("\nBased on the p-value, there is a statistically significant difference in 'taxa_relevante' between minority and non-minority groups.")
else:
    print("\nBased on the p-value, there is no statistically significant difference in 'taxa_relevante' between minority and non-minority groups.")

T-statistic: -3.416
P-value: 0.001

Based on the p-value, there is a statistically significant difference in 'taxa_relevante' between minority and non-minority groups.


In [ ]:
numerical_columns = simplificadas.select_dtypes(include=['number']).columns

columns_to_exclude = ['id', 'len_transcricao', 'len_transcricao_palavras', 'len_total', 'n_turnos', 'is_minoria']

columns_for_t_test = [col for col in numerical_columns if col not in columns_to_exclude]

print("Performing independent samples t-test for numerical columns:")
for col in columns_for_t_test:
    print(f"\n--- Column: {col} ---")
    minority_group_values = simplificadas[simplificadas['is_minoria']][col]
    non_minority_group_values = simplificadas[simplificadas['is_minoria'] == False][col]

    minority_group_values = minority_group_values.dropna()
    non_minority_group_values = non_minority_group_values.dropna()

    if len(minority_group_values) > 1 and len(non_minority_group_values) > 1:
        t_statistic, p_value = stats.ttest_ind(minority_group_values, non_minority_group_values)

        print(f"  T-statistic: {t_statistic:.3f}")
        print(f"  P-value: {p_value:.3f}")

        if p_value < 0.05:
            print(f"  Conclusion: There is a statistically significant difference in '{col}' between minority and non-minority groups.")
        else:
            print(f"  Conclusion: There is no statistically significant difference in '{col}' between minority and non-minority groups.")
    else:
        print(f"  Not enough data in one or both groups to perform t-test for '{col}'.")

Performing independent samples t-test for numerical columns:

--- Column: n_oradores ---
  T-statistic: -0.819
  P-value: 0.414
  Conclusion: There is no statistically significant difference in 'n_oradores' between minority and non-minority groups.

--- Column: len_ids_relevantes ---
  T-statistic: -1.436
  P-value: 0.153
  Conclusion: There is no statistically significant difference in 'len_ids_relevantes' between minority and non-minority groups.

--- Column: len_ids_protocolo ---
  T-statistic: 1.861
  P-value: 0.064
  Conclusion: There is no statistically significant difference in 'len_ids_protocolo' between minority and non-minority groups.

--- Column: len_ids_gestao_da_audiencia ---
  T-statistic: -1.135
  P-value: 0.258
  Conclusion: There is no statistically significant difference in 'len_ids_gestao_da_audiencia' between minority and non-minority groups.

--- Column: len_ids_digressao ---
  T-statistic: -0.707
  P-value: 0.480
  Conclusion: There is no statistically significan